In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/activities_final.csv", parse_dates=["date"])

# Dataset 1 — all runs, no Whoop features
strava_features = ["distance_km", "duration_sec", "elevation_m",
                   "avg_hr", "avg_speed", "pace_min_per_km",
                   "rolling_7d_km_daily", "rolling_28d_km_daily", "rolling_42d_km_daily",
                   "acwr_daily", "day_of_week", "month", "hour"]

df_strava = df[strava_features + ["date", "id"]].copy()
print(f"Strava dataset: {len(df_strava)} runs")

# Dataset 2 — runs with Whoop data
whoop_features = strava_features + ["recovery_score", "hrv_rmssd",
                                     "resting_hr", "sleep_performance_pct",
                                     "deep_sleep_min", "rem_sleep_min"]

df_whoop = df[df["recovery_score"].notna()][whoop_features + ["date", "id"]].copy()
print(f"Whoop dataset: {len(df_whoop)} runs")

Strava dataset: 509 runs
Whoop dataset: 235 runs


In [2]:
print(df.columns.tolist())

['date', 'id', 'name', 'distance_km', 'duration_sec', 'elevation_m', 'avg_hr', 'max_hr', 'avg_speed', 'pace_min_per_km', 'week', 'day_of_week', 'month', 'hour', 'time_of_day', 'days_since_last_run', 'rolling_7d_km', 'rolling_28d_km', 'rolling_42d_km', 'acwr', 'rolling_7d_km_daily', 'rolling_28d_km_daily', 'rolling_42d_km_daily', 'acwr_daily', 'date_day', 'recovery_score', 'hrv_rmssd', 'resting_hr', 'skin_temp_celsius', 'spo2_pct', 'sleep_performance_pct', 'sleep_duration_min', 'deep_sleep_min', 'rem_sleep_min', 'light_sleep_min']


## Model 1 — XGBoost Pace Predictor
Predicting average pace (min/km) from training load and activity features.
Baseline model — establishes performance ceiling for more complex models.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import numpy as np

# Target: pace
# Drop rows where target is missing
model1_features = ["distance_km", "elevation_m", "rolling_7d_km_daily", 
                   "rolling_28d_km_daily", "acwr_daily", 
                   "day_of_week", "month", "hour"]

df_model1 = df_strava[model1_features + ["pace_min_per_km"]].dropna()

X = df_model1[model1_features]
y = df_model1["pace_min_per_km"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 402
Test samples: 101


In [4]:
# Filter to easy runs only (avg_hr < 155 or no HR — use distance as proxy)
df_model1 = df_strava[model1_features + ["pace_min_per_km", "avg_hr"]].dropna(
    subset=model1_features + ["pace_min_per_km"]
)

# Only easy runs where HR is available and below 155
df_model1_easy = df_model1[
    (df_model1["avg_hr"].isna()) | (df_model1["avg_hr"] < 155)
]

print(f"Easy runs for modelling: {len(df_model1_easy)}")

X = df_model1_easy[model1_features]
y = df_model1_easy["pace_min_per_km"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Easy runs for modelling: 419


In [5]:
model1_features = ["distance_km", "elevation_m", "rolling_7d_km_daily",
                   "rolling_28d_km_daily", "acwr_daily",
                   "day_of_week", "month", "hour", "avg_hr"]  # add avg_hr

df_model1 = df_strava[model1_features + ["pace_min_per_km"]].dropna()

X = df_model1[model1_features]
y = df_model1["pace_min_per_km"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.3f} min/km")
print(f"R²: {r2:.3f}")
print(f"Mean error in seconds/km: {mae * 60:.1f}s")

MAE: 0.353 min/km
R²: 0.472
Mean error in seconds/km: 21.2s


In [7]:
import plotly.express as px
importance = pd.DataFrame({
    "feature": model1_features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=True)

fig = px.bar(importance, x="importance", y="feature", 
             orientation="h",
             title="XGBoost Feature Importance — Pace Predictor")
fig.show()

### Model 1 Results
- **R² = 0.472** — model explains 47% of pace variance on easy runs
- **MAE = 21.2 sec/km** — average prediction error
- **Key finding:** Heart rate is the dominant predictor, followed by chronic training load and elevation
- **Limitation:** Model only valid for easy runs (HR < 155bpm)

## Model 3 — LSTM HR Efficiency Trend
Using a Long Short-Term Memory network to learn the temporal pattern of cardiovascular fitness from HR efficiency on easy runs.
Unlike XGBoost which sees each run independently, the LSTM remembers the sequence — a dip after high mileage means something different than a dip after doing nothing.

In [9]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler

# Easy runs with HR
df_easy = df[
    (df["avg_hr"].notna()) & 
    (df["avg_hr"] < 155)
].copy()

# Compute HR efficiency
df_easy["speed_m_per_min"] = (df_easy["distance_km"] * 1000) / (df_easy["duration_sec"] / 60)
df_easy["hr_efficiency"] = df_easy["speed_m_per_min"] / df_easy["avg_hr"]
df_easy = df_easy[df_easy["hr_efficiency"] < 1.8].copy()
df_easy = df_easy.sort_values("date").reset_index(drop=True)

print(f"Easy runs: {len(df_easy)}")

# Scale
scaler = MinMaxScaler()
values = scaler.fit_transform(df_easy[["hr_efficiency"]])

# Create sequences
SEQ_LEN = 4

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(values, SEQ_LEN)

# Train/test split
split = len(X) - 20
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training sequences: {len(X_train)}")
print(f"Test sequences: {len(X_test)}")

Easy runs: 225
Training sequences: 201
Test sequences: 20


In [ ]:
model_lstm = keras.Sequential([
    keras.layers.LSTM(16, input_shape=(SEQ_LEN, 1)),
    keras.layers.Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.summary()